# LHRM Baseline — Full Training (RUN A)

**GATES — read before running:**
1. DR1 (1000-step diagnostic) must have PASSED and been recorded in `docs/CHATGPT_PROJECT_CHECKPOINT.md`.
2. FREEZE-B=L2 must be set and recorded.
3. This notebook runs Cell 1 (env check), Cell 2 (provenance), Cell 3 (preflight), then Cell 4 (training).
4. Cell 4 is labelled `[LAUNCH CELL]`. Do **NOT** run it unless all gates are satisfied.
5. Test split is **never** evaluated during training (A1). Use Cell 5 after training completes.

**Architecture:** L2 (scaled dot-product; `S = (q @ k.T) / sqrt(768)`) — paper-adjacent, not paper-exact.

**Checkpoint strategy (A9):** Rolling only — `latest.pt` and `previous.pt`. No per-epoch files by default.

In [ ]:
# ============================================================
# Cell 1 — Environment check
# ============================================================
import subprocess, sys, os, json, hashlib, time

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())
    return r.returncode

print('=== GPU ===')
sh('nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader')

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('Memory:', torch.cuda.get_device_properties(0).total_memory // (1024**2), 'MB')
else:
    raise RuntimeError('No CUDA device — training requires GPU')

print()
sh('python --version')
sh('pip show transformers | head -3')
sh('pip show scikit-learn | head -3')

In [ ]:
# ============================================================
# Cell 2 — Bundle extraction and provenance verification
# ============================================================
import zipfile, glob

# --- Locate bundle ---
BUNDLE_SEARCH = [
    '/kaggle/input/lhrm-bundle/lhrm_bundle_v3.zip',
    '/kaggle/input/lhrm-bundle/*.zip',
    '/kaggle/working/lhrm_bundle*.zip',
]
bundle_path = None
for pattern in BUNDLE_SEARCH:
    matches = glob.glob(pattern)
    if matches:
        bundle_path = matches[0]
        break
if bundle_path is None:
    raise FileNotFoundError('Bundle ZIP not found. Upload lhrm_bundle_v3.zip as Kaggle dataset.')
print('Bundle:', bundle_path)

WORK = '/kaggle/working'
if not os.path.isdir(os.path.join(WORK, 'run_baseline.py')):
    with zipfile.ZipFile(bundle_path, 'r') as zf:
        zf.extractall(WORK)
    print('Extracted to:', WORK)
else:
    print('Bundle already extracted.')

PROJ = WORK
sys.path.insert(0, os.path.join(PROJ, 'src'))

# --- Locate BERT ---
BERT_SEARCH = [
    '/kaggle/input/bert-base-uncased',
    '/kaggle/input/google-bert-base-uncased',
    '/kaggle/input/bertbaseuncased',
]
BERT_PATH = None
for p in BERT_SEARCH:
    if os.path.isdir(p):
        BERT_PATH = p
        break
if BERT_PATH is None:
    raise FileNotFoundError('Official bert-base-uncased not found in /kaggle/input/. '
                             'Add the official HuggingFace model as a Kaggle dataset.')
print('BERT path:', BERT_PATH)

# Verify vocab_size
from transformers import BertTokenizer
tok = BertTokenizer.from_pretrained(BERT_PATH)
assert tok.vocab_size == 30522, f'Expected 30522, got {tok.vocab_size}'
print('BERT vocab_size: 30522 OK')

# --- Locate AAPD ---
AAPD_SEARCH = [
    '/kaggle/input/aapd-dataset/AAPD',
    '/kaggle/input/aapd/AAPD',
    '/kaggle/input/aapd',
]
AAPD_DIR = None
for p in AAPD_SEARCH:
    if os.path.isfile(os.path.join(p, 'train.csv')):
        AAPD_DIR = p
        break
if AAPD_DIR is None:
    raise FileNotFoundError('AAPD dataset not found. Upload the Zenodo AAPD dataset.')
print('AAPD dir:', AAPD_DIR)

# CSV integrity
EXPECTED_CSV_HASHES = {
    'train.csv': 'e7e73064385f78fe092edd47d4e4059c35875a1ea2a95859ad3cde091ed9af95',
    'dev.csv':   'a85f612d4c1cb2be6d1de94cbcd9468cba97d221f826014264f4d279c82e220b',
}
for fname, expected in EXPECTED_CSV_HASHES.items():
    fpath = os.path.join(AAPD_DIR, fname)
    if not os.path.exists(fpath):
        raise FileNotFoundError(f'{fname} not found in {AAPD_DIR}')
    h = hashlib.sha256(open(fpath, 'rb').read()).hexdigest()
    if h != expected:
        raise ValueError(f'{fname} hash mismatch: got {h}')
    print(f'  {fname}: SHA256 PASS')

# --- Label order ---
LABEL_ORDER_EXPECTED = 'a3d662f3920259a94ed61d83f98f21cabe7f908d98a5662ccd9adc2413cc000b'
proc_dir = os.path.join(PROJ, 'data', 'processed', 'AAPD')
if not os.path.isdir(proc_dir):
    os.makedirs(proc_dir, exist_ok=True)
    # Run prepare to extract
    os.environ['LHRM_AAPD_DIR'] = AAPD_DIR
    sh(f'python {PROJ}/run_baseline.py prepare --config {PROJ}/configs/final/aapd_baseline.yaml')
labels_json = os.path.join(proc_dir, 'labels.json')
if os.path.exists(labels_json):
    label_order = json.load(open(labels_json))
    lo_hash = hashlib.sha256(json.dumps(label_order).encode()).hexdigest()
    if lo_hash != LABEL_ORDER_EXPECTED:
        raise ValueError(f'label_order hash mismatch: got {lo_hash}')
    print(f'label_order ({len(label_order)} labels): SHA256 PASS')
else:
    print('WARNING: labels.json not found in proc_dir. Run prepare first.')

print('\n=== PROVENANCE VERIFIED ===')

In [ ]:
# ============================================================
# Cell 3 — Architecture gate check (FREEZE-B must be set)
# ============================================================
import json, os

PROJ = '/kaggle/working'
CHECKPOINT_DOC = os.path.join(PROJ, 'docs', 'CHATGPT_PROJECT_CHECKPOINT.md')
FREEZE_PATH = os.path.join(PROJ, 'configs', 'final', 'CONFIG_FREEZE.json')

# Check CONFIG_FREEZE.json exists
if not os.path.exists(FREEZE_PATH):
    raise RuntimeError(
        f'CONFIG_FREEZE.json not found at {FREEZE_PATH}.\n'
        'Run: python run_baseline.py freeze --config configs/final/aapd_baseline.yaml '
        '--out configs/final/CONFIG_FREEZE.json\n'
        'Then re-upload bundle.')
freeze = json.load(open(FREEZE_PATH))
variant = freeze.get('architecture') or freeze.get('variant')
print('CONFIG_FREEZE architecture:', variant)
if variant == 'TBD' or not variant:
    raise RuntimeError('CONFIG_FREEZE has architecture=TBD. FREEZE-B not set.')
if variant not in ('L', 'L2'):
    raise RuntimeError(f'Unknown architecture variant: {variant}. Must be L or L2.')
print(f'FREEZE-B = {variant} OK')

# Check DR1 approval recorded
if os.path.exists(CHECKPOINT_DOC):
    ckpt_text = open(CHECKPOINT_DOC).read()
    dr1_approved = 'DR1' in ckpt_text and ('APPROVED' in ckpt_text or 'PASS' in ckpt_text)
    print('DR1 approval in checkpoint doc:', dr1_approved)
    if not dr1_approved:
        print('WARNING: DR1 approval not found in CHATGPT_PROJECT_CHECKPOINT.md.')
        print('Ensure DR1 is recorded before launching full training.')
else:
    print('WARNING: CHATGPT_PROJECT_CHECKPOINT.md not found.')

print('\nRUN ID will be: RUN_A_' + variant)
RUN_ID = 'RUN_A_' + variant
print('\nGate check complete. Proceed to Cell 4 only if gates satisfied.')

In [ ]:
# ============================================================
# Cell 4 — [LAUNCH CELL] Full 15-epoch training
# DO NOT RUN unless Cell 3 gate check passed and DR1 APPROVED.
# ============================================================
import subprocess, sys, os

PROJ = '/kaggle/working'
BERT_PATH = '/kaggle/input/bert-base-uncased'  # adjust to actual mount
RUN_ID = 'RUN_A_L2'  # set from Cell 3 output
VARIANT = 'L2'       # must match FREEZE-B
CONFIG = os.path.join(PROJ, 'configs', 'final', 'aapd_baseline.yaml')
RESUME_FROM = None   # set to path of latest.pt to resume

cmd = [
    'python', '-u',  # -u: unbuffered stdout (A2: live progress)
    os.path.join(PROJ, 'run_baseline.py'),
    'train',
    '--config', CONFIG,
    '--run_id', RUN_ID,
    '--variant', VARIANT,
]
if RESUME_FROM:
    cmd += ['--resume_from', RESUME_FROM]

print('Command:', ' '.join(cmd))
print('Launching training... (stream output below)')
print('Screenshot evidence: first [TRAIN] line appears within ~250 steps (~75s on T4)')
print()

# Set offline BERT path
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=PROJ,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print('\n=== Training process exited with code:', proc.returncode, '===')
if proc.returncode != 0:
    raise RuntimeError('Training failed with exit code ' + str(proc.returncode))

In [ ]:
# ============================================================
# Cell 5 — [POST-TRAINING ONLY] Final test evaluation (A1)
# Run ONCE after training complete. Requires explicit authorization.
# ============================================================
import subprocess, os

PROJ = '/kaggle/working'
RUN_ID = 'RUN_A_L2'
FINAL_CKPT = os.path.join(PROJ, 'experiments', RUN_ID, 'checkpoints', 'final_epoch.pt')
CONFIG = os.path.join(PROJ, 'configs', 'final', 'aapd_baseline.yaml')

# Safety: check training complete
if not os.path.exists(FINAL_CKPT):
    raise FileNotFoundError(
        f'final_epoch.pt not found at {FINAL_CKPT}.\n'
        'Training must complete before test evaluation.')

# Check test predictions don't already exist
pred_path = os.path.join(PROJ, 'experiments', RUN_ID, 'predictions_test_raw.json')
if os.path.exists(pred_path):
    raise RuntimeError(
        'predictions_test_raw.json already exists.\n'
        'Test evaluation must run exactly once (A1).')

print('Running final test evaluation (ONCE)...')
r = subprocess.run([
    'python', '-u',
    os.path.join(PROJ, 'run_baseline.py'),
    'test_eval',
    '--config', CONFIG,
    '--run_id', RUN_ID,
    '--checkpoint', FINAL_CKPT,
], capture_output=False, text=True, cwd=PROJ)
print('Exit code:', r.returncode)

In [ ]:
# ============================================================
# Cell 6 — Package evidence ZIP (no checkpoints, no test data)
# ============================================================
import zipfile, os, glob, hashlib, json, time

PROJ = '/kaggle/working'
RUN_ID = 'RUN_A_L2'
EXP_DIR = os.path.join(PROJ, 'experiments', RUN_ID)
OUT_ZIP = f'/kaggle/working/{RUN_ID}_EVIDENCE_{time.strftime("%Y%m%d_%H%M%S")}.zip'

# Files to include (small; no checkpoints; no test predictions unless authorized)
INCLUDE_PATTERNS = [
    'train.log',
    'train_progress.jsonl',
    'train_history.json',
    'epoch_metrics.csv',
    'resolved_config.json',
    'SHA256SUMS.txt',
    'predictions_val.json',
    'val_per_label_metrics.json',
]
# Add FREEZE and DR approval docs
EXTRA_FILES = [
    os.path.join(PROJ, 'configs', 'final', 'CONFIG_FREEZE.json'),
    os.path.join(PROJ, 'docs', 'CHATGPT_PROJECT_CHECKPOINT.md'),
    os.path.join(PROJ, 'experiments', 'RUN_0_P1F', 'ARTIFACT_POINTER.json'),
]

with zipfile.ZipFile(OUT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for pattern in INCLUDE_PATTERNS:
        fpath = os.path.join(EXP_DIR, pattern)
        if os.path.exists(fpath):
            zf.write(fpath, os.path.join(RUN_ID, pattern))
            print('Added:', pattern)
    for fpath in EXTRA_FILES:
        if os.path.exists(fpath):
            zf.write(fpath, os.path.basename(fpath))
            print('Added:', os.path.basename(fpath))

# Compute ZIP SHA256
h = hashlib.sha256(open(OUT_ZIP, 'rb').read()).hexdigest()
print(f'\nEvidence ZIP: {OUT_ZIP}')
print(f'SHA256: {h}')
print(f'Size: {os.path.getsize(OUT_ZIP) / 1e6:.1f} MB')